# NTN scene classification from distilled LSPs

Reads the datasets written by `custom_drivers/scene_class_dataset.ipynb`
(`Results/scene_class/<dataset>/`), distils them (`scene_dataset.build_dataset`) and asks
whether per-scene LSP statistics classify rural / suburban / urban. Folds are grouped by
scene seed throughout. Only `feature` and `context` columns (elevation, carrier,
polarization pair) enter a model; labels, material draws and simulator truth never do.
Tables go to `Results/scene_class/analysis/`. Details: `docs/scene_classification.md`.

In [ ]:
import os, sys, warnings
from pathlib import Path
_starts = [Path(p).resolve().parent for p in [globals().get("__vsc_ipynb_file__")] if p]
_starts.append(Path.cwd().resolve())
REPO = next(d for s in _starts for d in (s, *s.parents, s / "Ray_Tracing") if (d / "src" / "utils").is_dir())
if str(REPO.parent) not in sys.path:
    sys.path.insert(0, str(REPO.parent))
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from Ray_Tracing.src.utils import scene_classes as scn
from Ray_Tracing.src.utils import scene_dataset as sd
from Ray_Tracing.src.utils import scene_classify as sc
from Ray_Tracing.src.utils.sensitivity import dist_metrics

ROOT = str(REPO / "Results" / "scene_class")
OUT  = os.path.join(ROOT, "analysis")
os.makedirs(OUT, exist_ok=True)
REBUILD = False    # True after new results land

DS = {}
for ds in ("pilot", "base", "hard", "matctl", "base_mixedpol"):
    d = os.path.join(ROOT, "dataset", ds)
    if REBUILD or not os.path.exists(os.path.join(d, "scenes.pkl")):
        if not os.path.isdir(os.path.join(ROOT, ds)):
            continue
        sd.build_dataset(ROOT, [ds])
    DS[ds] = (pd.read_pickle(os.path.join(d, "links.pkl")), pd.read_pickle(os.path.join(d, "scenes.pkl")))
MAIN = "base" if "base" in DS else "pilot"
links, scenes = DS[MAIN]
print({k: (len(l), len(s)) for k, (l, s) in DS.items()}, "-> main:", MAIN)

## 1 - Class sanity: P_LoS and LSP marginals

In [ ]:
pl = links.pivot_table(index="elevation", columns=["frequency", "scene_class"], values="is_los")
display(pl.round(3))
fig, ax = plt.subplots(figsize=(6, 3.5))
for cls in scn.CLASSES:
    g = links[(links.scene_class == cls) & (links.frequency == links.frequency.min())].groupby("elevation").is_los.mean()
    l, = ax.plot(g.index, g.values, "o-", label=f"{cls} RT")
    ax.plot(g.index, scn.plos_chiu(g.index, cls), "--", color=l.get_color(), label=f"{cls} Chiu (27)")
ax.set(xlabel="elevation [deg]", ylabel="P_LoS (RT LoS flag)"); ax.legend(fontsize=7); plt.show()

In [ ]:
SHOW = ["f_K_rom_dB", "f_lgDS_all_mean", "f_lgASA_all_mean", "f_lgZSA_all_mean",
        "f_excess_loss_dB_all_mean", "f_lg_doppler_rms_hz_all_mean"]
f0 = scenes.frequency.min()
fig, axs = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)
for ax, col in zip(axs.flat, SHOW):
    for cls in scn.CLASSES:
        g = scenes[(scenes.scene_class == cls) & (scenes.frequency == f0)].groupby("elevation")[col]
        ax.fill_between(g.median().index, g.quantile(0.1), g.quantile(0.9), alpha=0.2)
        ax.plot(g.median().index, g.median(), "o-", ms=3, label=cls)
    ax.set_title(col, fontsize=9)
axs[0, 0].legend(fontsize=7)
for ax in axs[1]:
    ax.set_xlabel("elevation [deg]")
fig.suptitle(f"per-scene medians and 10-90 % bands, {f0 / 1e9:g} GHz"); plt.tight_layout(); plt.show()

In [ ]:
# Per-UE LSPs against the matching TR 38.811 table (class -> scenario).
REF = {"rural": "Rural", "suburban": "Suburban", "urban": "Urban"}
dm = pd.concat([dist_metrics(links[links.scene_class == c].assign(variant=c), REF[c])
                for c in scn.CLASSES], ignore_index=True)
dm.to_csv(os.path.join(OUT, f"{MAIN}_dist_38811.csv"), index=False)
display(dm.pivot_table(index=["state", "lsp"], columns=["variant"], values="w1").round(3))

## 2 - Classification

Per-scene feature vectors (the scene's 20 UEs distilled), one row per (scene, elevation,
carrier). Gradient boosting handles missing states (no LoS UE → NaN LoS statistics);
the logistic model imputes medians.

In [ ]:
RES = {k: sc.grouped_cv(scenes, k) for k in ("hgb", "logreg")}
CTRL = sc.grouped_cv(scenes, "hgb", shuffle_labels=True)
print({k: round(sc.overall_accuracy(r), 3) for k, r in RES.items()},
      "shuffled-label control:", round(sc.overall_accuracy(CTRL), 3))
acc = pd.concat([sc.accuracy_by(r).assign(model=k) for k, r in RES.items()])
acc.to_csv(os.path.join(OUT, f"{MAIN}_accuracy_by_elev.csv"), index=False)
display(acc.pivot_table(index="elevation", columns=["model", "frequency"], values="accuracy").round(3))
display(sc.confusion(RES["hgb"]).round(2))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
for (k, f), g in acc.groupby(["model", "frequency"]):
    axs[0].plot(g.elevation, g.accuracy, "o-", label=f"{k} {f / 1e9:g} GHz")
axs[0].axhline(1 / 3, color="k", lw=0.8, ls=":")
axs[0].set(xlabel="elevation [deg]", ylabel="accuracy", ylim=(0, 1.02)); axs[0].legend(fontsize=7)
cm = sc.confusion(RES["hgb"])
axs[1].imshow(cm, vmin=0, vmax=1, cmap="Blues")
axs[1].set(xticks=range(3), yticks=range(3), xticklabels=cm.columns, yticklabels=cm.index,
           xlabel="predicted", ylabel="true")
for i in range(3):
    for j in range(3):
        axs[1].text(j, i, f"{cm.iat[i, j]:.2f}", ha="center", va="center")
plt.tight_layout(); plt.show()

## 3 - How many UEs does a scene need?

In [ ]:
rows = []
for n in (1, 2, 5, 10, 20):
    s_n = sd.scene_features(links, n_ue=n, seed=n)
    r = sc.grouped_cv(s_n, "hgb")
    rows += [dict(n_ue=n, **a) for a in sc.accuracy_by(r, ["frequency"]).to_dict("records")]
nue = pd.DataFrame(rows)
nue.to_csv(os.path.join(OUT, f"{MAIN}_accuracy_vs_n_ue.csv"), index=False)
display(nue.pivot(index="n_ue", columns="frequency", values="accuracy").round(3))

## 4 - Which LSPs carry the class?

In [ ]:
imp = sc.lsp_importance(scenes)
imp.to_csv(os.path.join(OUT, f"{MAIN}_lsp_importance.csv"))
imp.plot.barh(figsize=(5, 3.5), title="permutation importance (accuracy drop), summed per LSP")
plt.gca().invert_yaxis(); plt.show()

## 5 - Robustness to the link nuisances

Train on circular polarization pairs, test on linear ones (and back). The pair is a context
column, so this tests whether the LSP-to-class map survives an unseen polarization.

In [ ]:
# base is RHCP/RHCP only since 2026-10-19; the per-UE MLP version (section 7) uses base_mixedpol.
CP = scenes.pol_pair.isin(["RHCP/RHCP", "RHCP/LHCP"])
LIN = scenes.pol_pair.isin(["V/V", "H/H"])
if CP.any() and LIN.any():
    cols = [c for c in sd.feature_columns(scenes) if c != "pol_pair"]
    xfer = {"CP -> linear": sc.overall_accuracy(sc.train_test(scenes[CP], scenes[LIN], cols=cols)),
            "linear -> CP": sc.overall_accuracy(sc.train_test(scenes[LIN], scenes[CP], cols=cols)),
            "in-distribution (CV)": sc.overall_accuracy(RES["hgb"])}
    display(pd.Series(xfer).round(3))

## 6 - Geometry vs materials

`matctl` draws every class's materials from the pooled anchor weights, so materials carry
no class information there: its accuracy is what geometry alone buys. Within `base`,
accuracy conditioned on the drawn anchors shows which scenes the materials rescue or confuse.

In [ ]:
cmp = {"base (geometry + materials)": sc.overall_accuracy(RES["hgb"])}
if "matctl" in DS:
    cmp["matctl (geometry only)"] = sc.overall_accuracy(sc.grouped_cv(DS["matctl"][1], "hgb"))
if "hard" in DS:
    cmp["hard (p = 0.30)"] = sc.overall_accuracy(sc.grouped_cv(DS["hard"][1], "hgb"))
display(pd.Series(cmp).round(3))
r = RES["hgb"].assign(ok=lambda d: d.pred == d.label)
for role in ("roof", "ground", "wall"):
    t = r.groupby(["scene_class", f"{role}_anchor"]).ok.agg(accuracy="mean", n="size")
    t.to_csv(os.path.join(OUT, f"{MAIN}_accuracy_by_{role}_anchor.csv"))
    display(t.round(3))

## 7 - MLP on elevation curves

The MLP classifier lives in [`scene_mlp_classifier.ipynb`](scene_mlp_classifier.ipynb). Its input is one UE's
whole elevation sweep (each LSP at 10–80°) plus carrier and pol pair, and it runs on any results
directory. This notebook keeps the scene-level HGB / logit baselines.